# 05 — Authenticate, then query

**Access pattern: a credentialed API with a catalogue.** Everything so far was
anonymous. This one is not.

The Copernicus Marine Service is the only source in the workshop that needs an account.
It is also the only one that supplies **currents** — Argo has temperature and salinity
but no velocity, and ERDDAP has surface temperature only — which is why the account is
load-bearing rather than optional.

**If you have not set up an account, this notebook still runs.** It detects that and
teaches the parts that do not need credentials. See section 1.

In [1]:
import sys
from pathlib import Path

# notebooks/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
for p in (Path.cwd(), Path.cwd() / ".." / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S

# A real requests.Session, with a disk cache. Everything you use below -- .get,
# .text, .content, .json, .status_code, .raise_for_status, params= -- is
# ordinary `requests`. The cache is invisible at the call site.
http = _fetch.session()

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

_fetch.OFFLINE and print("offline mode: serving prefetched responses")

False

## 1. Check first, fail gracefully

Account provisioning is slow and somebody always arrives with an unverified address. The
honest design is to detect that and keep teaching, not to raise a `Traceback` at a room
of thirty people.

In [2]:
import os
import shutil
import subprocess
from pathlib import Path

from ocean_data_workshop.credentials import DOTENV, child_env, describe, load

has_cli = shutil.which("copernicusmarine") is not None
creds = load()
where = (creds["source"] if (creds["complete"] or creds["login_blob_exists"]) else "NONE")

print(f"  copernicusmarine CLI : {'found' if has_cli else 'NOT FOUND'}")
print(f"  .env                 : {'found' if creds['dotenv_exists'] else 'not found'}")
print(f"  credentials          : {where}")
print(describe())

READY = has_cli and (creds["complete"] or creds["login_blob_exists"])

if READY:
    print("  -> running the live sections below.")
else:
    print("  -> NOT READY. Sections 3-5 are skipped; 2 and 4 still teach something.")
    print()
    print("  To set this up (takes a few minutes, and an email verification):")
    print("    1. Register: https://data.marine.copernicus.eu/register")
    print("    2. copernicusmarine login")
    print()
    print("  Either:")
    print(f"    1. put them in {DOTENV.name} (gitignored, and the loader reads it):")
    print("         COPERNICUSMARINE_SERVICE_USERNAME=you@example.com")
    print("         COPERNICUSMARINE_SERVICE_PASSWORD=...")
    print("       then:  chmod 600 .env")
    print("    2. or run `copernicusmarine login`, which is interactive and writes")
    print("       the result to ~/.copernicusmarine/")
    print()
    print("  The variable names are not guessable: COPERNICUS_USERNAME and")
    print("  COPERNICUS_PASSWORD are silently ignored. No credential is committed.")

  copernicusmarine CLI : found
  .env                 : found
  credentials          : .env
  credentials: present, from .env
  -> running the live sections below.


## 2. The catalogue, and a 170 MB mistake

Every Copernicus dataset has a metadata catalogue. Fetch it *filtered*, never bare.

| command | result |
|---|---|
| `describe` with no filter | **170 MB** of JSON, minutes of download |
| `describe --dataset-id <id>` | a few KB |

This is the single most common way this CLI is misused, and it looks like a network
problem rather than a syntax problem.

In [3]:
# Show the flag, not the result -- the bare call is deliberately NOT made here.
cmd = ["copernicusmarine", "describe", "--dataset-id", S.GLORYS_DATASET]
print("  the right way:")
print("   ", " ".join(cmd))
print()
print("  the 170 MB way, which you should never run by accident:")
print("    copernicusmarine describe")
print()
print("  Other verified flags, because the CLI is not internally consistent:")
print("    --end-datetime   NOT --stop-datetime  (the latter is rejected)")
print("    --variable       pass explicitly; the advertised variable count is 0")

  the right way:
    copernicusmarine describe --dataset-id cmems_mod_glo_phy_my_0.083deg_P1D-m

  the 170 MB way, which you should never run by accident:
    copernicusmarine describe

  Other verified flags, because the CLI is not internally consistent:
    --end-datetime   NOT --stop-datetime  (the latter is rejected)
    --variable       pass explicitly; the advertised variable count is 0


### ⚠️ Trap — the advertised variable count is `0`

`copernicusmarine describe` reports the number of variables in a dataset as **zero**,
for datasets that plainly have variables. A correct dataset looks empty.

The consequence is specific: a script that discovers variables by asking the catalogue
gets nothing, and quietly requests no data. Do not use the summary to build a request —
name the variables you want.

In [4]:
VARIABLES = ["thetao", "so", "uo", "vo"]   # temperature, salinity, u, v
print("  the request is built from variables WE name, not from what the catalogue says:")
print("   ", ", ".join(VARIABLES))
print()
print("  GLORYS short names, and what they mean:")
for v, meaning in zip(VARIABLES, ["potential temperature", "salinity",
                                  "eastward current", "northward current"]):
    print(f"    {v:8} {meaning}")
print()
print("  Our own project renames these to temperature/salinity/u_eastward/v_northward,")
print("  so the rest of the code never has to know the vendor's short names.")

  the request is built from variables WE name, not from what the catalogue says:
    thetao, so, uo, vo

  GLORYS short names, and what they mean:
    thetao   potential temperature
    so       salinity
    uo       eastward current
    vo       northward current

  Our own project renames these to temperature/salinity/u_eastward/v_northward,
  so the rest of the code never has to know the vendor's short names.


## 3. A subset request

Bounding the request in **all** six dimensions is what makes it fast. The dataset is
global at 1/12° with 50 levels and daily resolution from 1993; the box is 0.22° square
and 60 m deep, for a two-year window.

In [5]:
from ocean_data_workshop.config import OCEAN_BOX

box = OCEAN_BOX
subset = [
    "copernicusmarine", "subset",
    "--dataset-id", S.GLORYS_DATASET,
    "--variable", "thetao",
    "--minimum-longitude", str(box["min_longitude"]),
    "--maximum-longitude", str(box["max_longitude"]),
    "--minimum-latitude",  str(box["min_latitude"]),
    "--maximum-latitude",  str(box["max_latitude"]),
    "--minimum-depth",     str(box["min_depth"]),
    "--maximum-depth",     str(box["max_depth"]),
    "--start-datetime", "2019-01-01T00:00:00",
    "--end-datetime",   "2021-05-01T00:00:00",
    "--output-directory", "data/glorys",
    "--file-format", "netcdf",
]
for part in subset:
    print("   ", part)
print()
print("  Six bounds plus a time window. Omit any one and the request is global:")
print("  that is the difference between a 30-second call and a stalled workshop.")

    copernicusmarine
    subset
    --dataset-id
    cmems_mod_glo_phy_my_0.083deg_P1D-m
    --variable
    thetao
    --minimum-longitude
    -122.2
    --maximum-longitude
    -121.98
    --minimum-latitude
    36.6
    --maximum-latitude
    36.82
    --minimum-depth
    0.0
    --maximum-depth
    60.0
    --start-datetime
    2019-01-01T00:00:00
    --end-datetime
    2021-05-01T00:00:00
    --output-directory
    data/glorys
    --file-format
    netcdf

  Six bounds plus a time window. Omit any one and the request is global:
  that is the difference between a 30-second call and a stalled workshop.


In [6]:
if READY:
    print("  running the live subset -- this is the slow cell in the notebook...")
    # env=child_env() is NOT optional. The CLI reads credentials from the environment,
    # and a plain subprocess.run inherits this process's environment -- which does not
    # have them until child_env() loads .env. Without it the request fails with
    # "Dataset not found", which reads like a wrong dataset id and is not.
    r = subprocess.run(subset + ["--overwrite"], capture_output=True, text=True,
                       timeout=1800, env=child_env())
    print("  exit:", r.returncode)
    out = (r.stdout or r.stderr or "")
    print(out[-600:])
    if r.returncode != 0 and "not found" in out.lower():
        print()
        print("  'Dataset not found' means the dataset-id STRING was wrong, not that")
        print("  your account is. The form the CLI accepts is the CMEMS product code:")
        print(f"    {S.GLORYS_DATASET}")
        print("  and you must pass env=child_env() so the CLI receives your credentials.")
else:
    print("  SKIPPED -- no credentials on this machine.")
    print()
    print("  What you would have got, from the project's own loaded copy:")
    print("    30 days x 19 levels over 0.5-55.8 m, surface 15.62 C")
    print("    2019-2021: 16,188 rows in ocean_profile_daily (needs `make db-ocean`)")

  running the live subset -- this is the slow cell in the notebook...


  exit: 0
{
  "file_size": "0.21 MB",
  "data_transfer_size": "81.33 MB",
  "status": "000",
  "message": "The request was successful."
}



### ⚠️ Trap — the Product ID and the dataset id are different strings

This one produced a long and completely wrong detour, so it is worth telling properly.

Copernicus calls this product **GLOBAL_MULTIYEAR_PHY_001_030** on its website, and that
is the string you find by searching. `copernicusmarine` does **not** accept it. Both forms
were run against the live service, same credentials, same day:

| `--dataset-id` | result |
|---|---|
| `GLOBAL_MULTIYEAR_PHY_001_030` | `ERROR - Dataset not found` |
| `cmems_mod_glo_phy_my_0.083deg_P1D-m` | 200, file downloaded |

The working form is the CMEMS **product code**, and it is already right in
`src/ocean_data_workshop/config.py`. A constant written separately in `_sources.py` had it wrong —
so the loader worked while the notebook did not, from the same repository, on the same
machine, with the same credentials.

### ⚠️ Trap — and the error message sends you the wrong way

```
ERROR - Dataset not found: GLOBAL_MULTIYEAR_PHY_001_030
Please check that the dataset exists and the input datasetID is correct.
```

That advice is confidently wrong. I read it as a broken login, wrote it up as a trap
about stale credentials, and then tried four more dataset IDs before going back to read
the code that was already working.

Two things to take from it:

1. **`Dataset not found` is a statement about the string you passed, not about the
   product.** Check the failing command's own argv against a *working* one before
   theorising about your account.
2. **When two entry points disagree, diff them.** `glorys.fetch` succeeded while the
   notebook failed, and the difference was one argument. That comparison would have taken
   a minute and would have been conclusive.

The one thing I got right by accident, and which is still true: `login` reads these
environment variables, and the names are not the obvious ones.

```
COPERNICUSMARINE_SERVICE_USERNAME
COPERNICUSMARINE_SERVICE_PASSWORD
```

`COPERNICUS_USERNAME` / `COPERNICUS_PASSWORD` are silently ignored — no error, the
variables are simply never read. Check `copernicusmarine login --help` rather than
guessing.

## 4. Why a fallback ladder is not indecision

GLORYS is the only source for currents, and it is the only one that can fail for a
reason no amount of retrying fixes: **you do not have an account.**

That is a different failure from a timeout, and it should be planned for rather than
hoped against. This project therefore verified ten sources and built a ladder:

| tier | sources | if GLORYS is unavailable |
|---|---|---|
| 1 | GLORYS, Argo, ERDDAP SST, NDBC | — |
| 2 | WOA23 climatology, GEBCO bathymetry, HYCOM | temperature, salinity, currents from other providers |
| 3 | OBIS, Orcasound, ShipsEar, Watkins | biological and acoustic context |

A workshop that depends on one credentialed source is a workshop with a single point of
failure. This is the same reason the notebooks degrade to cache rather than raising: the
design assumption throughout is that *something* will be unavailable.

## 5. Handling credentials

Two rules, both learned the hard way:

1. **Never commit them.** This project keeps credentials in `~/.copernicusmarine/` and a
   gitignored `.env`, and no credential has ever been committed.
2. **Never share a laptop that has them.** A cached Copernicus login is a persistent
   credential on someone else's machine. This is the reason the workshop's prefetch
   deliberately does **not** cache anything from this notebook — every other source is
   anonymous, and caching those is harmless.

In [7]:
print("  readiness:", "LIVE" if READY else "SKIPPED (no credentials)")
print()
print("  For reference, the dataset this notebook targets:")
print("   ", S.COPERNICUS_HOME)
print()
print("  Product page, dataset id, and the catalogue of every variable are all public")
print("  and need no account -- it is only the data download that is credentialed.")

  readiness: LIVE

  For reference, the dataset this notebook targets:
    https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/description

  Product page, dataset id, and the catalogue of every variable are all public
  and need no account -- it is only the data download that is credentialed.
